### MiddleWear
#### Provides way to more tightly controll what happen inside the agent. middleware is useful for the following:
##### 1. Tracking agent behaviour with logging, analysis and debug.
##### 2. Transforming prompt, tool selection and output formatting.
##### 3. Adding retries, fallback, and early terminating logic.
##### 4. Applying rate limits, guadrials, and PII detection.

In [ ]:
## Gemini model test
import os
from langchain.chat_models import init_chat_model
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
model = init_chat_model("google_genai:gemini-2.5-flash")

In [12]:
model

ChatGoogleGenerativeAI(output_version=None, profile={'name': 'Gemini 2.5 Flash', 'release_date': '2025-03-20', 'last_updated': '2025-06-05', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), location=None, model='gemini-2.5-flash', client=<google.genai.client.Client object at 0x000001A36CE73A10>, default_metadata=(), model_kwargs={})

In [13]:
from langchain_groq import ChatGroq
llm = ChatGroq(
    model="llama-3.3-70b-versatile"
)

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")

### Summarize Middleware
It automatcally summaries conversation history when appraching token limits, preserving recent messages while compressing older context. Summaries is useful for the following :
- long-running conversations that exceed context windows.
- Multi-turn dialouge with extensive history.
- Applications where preserving full conversation context matters. 

In [14]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import SystemMessage, HumanMessage

#message based summariazation
agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("messages", 10),
            keep=("messages", 4)
        )
    ]
)

In [15]:
### run with thread id
congif={"configurable": {"thread_id":"test-1"}}

In [16]:
question = [
    "what is 2+2?",
    "what is 10+5?",
    "what is 100/2?",
    "what is 15-7?",
    "what is 3*3?",
    "what is 4+4?"
]

for q in question:
    response = agent.invoke({"messages":[HumanMessage(content=q)]}, config=congif)
    print(f"messages : {response}")
    print(f"length of messages {len(response["messages"])}")

messages : {'messages': [HumanMessage(content='what is 2+2?', additional_kwargs={}, response_metadata={}, id='180ee137-9189-4bd3-ad9a-b8e835721097'), AIMessage(content='2 + 2 = 4', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019ee394-9d14-79e1-a872-31de5851ed95-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 23, 'total_tokens': 31, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 16}})]}
length of messages 2
messages : {'messages': [HumanMessage(content='what is 2+2?', additional_kwargs={}, response_metadata={}, id='180ee137-9189-4bd3-ad9a-b8e835721097'), AIMessage(content='2 + 2 = 4', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019ee394-9d14-79e1-a872-31de5851ed95

### Token Size

In [25]:
from langchain_core.tools import tool

@tool
def search_hotles(city: str)-> str:
    """search hotels - return long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5star, $350/night, spa, pool, gym
    2. city Inn - 4 star, $180/night, business center
    3.Budget stay - 3 star, $75/night, free wifi"""

agent = create_agent(
    model=llm,
    tools=[search_hotles],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=("tokens",550),
            keep=("tokens", 200 )
        )
    ]
)

In [26]:
config = {"configurable": {"thread_id":"test-1"}}

In [27]:
#token counter (appoximate)
def count_token(messages):
    total_char = sum(len(str(m.content)) for m in messages)
    return total_char // 4 # 4 chars = 1 token

In [28]:
cities = ["Paris", "london", "tokyo", "New york", "dubai", "singapore"]

for city in cities:
    response = agent.invoke({"messages":[HumanMessage(content=f"Find hotels in {city}")]},
                            config=config
                            )
    tokens = count_token(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~296 tokens, 10 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='e2ad30f2-a582-4a9d-b8e4-be44480a430e'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'ba0p5qvaa', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotles'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 226, 'total_tokens': 241, 'completion_time': 0.049252312, 'completion_tokens_details': None, 'prompt_time': 0.057004262, 'prompt_tokens_details': None, 'queue_time': 0.054363238, 'total_time': 0.106256574}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ee3a4-590c-7c32-a157-fcb05c8b6376-0', tool_calls=[{'name': 'search_hotles', 'args': {'city': 'Paris'}, 'id': 'ba0p5qvaa', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metad

### Fraction
for using fraction trigger in summarization we need to do this changes only.
- trigger=["fraction", 0.005] # 0.5% of llm context
- keep=("fraction", 0.002) # 0.2% of llm context approx 256 tokens

### Human in the loop middleware

Pause agent execution for human approval, editing, or rejection of tool calls before they execute. human-in-the loop is useful for the following:

- high stake operations requirements human appoval (e g. database writes, financial transactions).
- compilances workflow where humans oversight is mandatory.
- Long - running conversation where human feedback guides the agent.

In [30]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email(email_id: str) -> str:
    """Mock fucntion to read email id """
    return f"Email content for ID: {email_id}"

def send_email_tool(receipt: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"email send to {receipt} with the subject {subject}"
 

In [33]:
agent = create_agent(
    model=llm,
    checkpointer=InMemorySaver(),
    tools=[read_email, send_email_tool],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                 "send_email_tool":{
                     "allowed_decisions":["approved", "edit", "reject"]
                 },
                 "read_email_tool":False,

            }
        )
    ]
)

In [34]:
config = {"configurable":{"thread_id": "test-approve"}}

result = agent.invoke(
    {"messages":[HumanMessage(content="send email to devendra@gmail.com with the subject 'hello' and body 'how are you?'")]},
    config=config
)

In [35]:
result

{'messages': [HumanMessage(content="send email to devendra@gmail.com with the subject 'hello' and body 'how are you?'", additional_kwargs={}, response_metadata={}, id='a55f6b22-5229-4527-b6f9-94cce9ce508d'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'hq5qkdd1h', 'function': {'arguments': '{"body":"how are you?","receipt":"devendra@gmail.com","subject":"hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 309, 'total_tokens': 337, 'completion_time': 0.071944898, 'completion_tokens_details': None, 'prompt_time': 0.053003131, 'prompt_tokens_details': None, 'queue_time': 0.497111869, 'total_time': 0.124948029}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_0761e44d7b', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ee42b-7e88-73a2-9c6d-e90c01dcc6b2-0', tool_calls=[{'name': 'send_email_tool',

In [ ]:
#step 2: approve
from langgraph.types import Command

if "__interrupt__" in result:
    print("paused! Approved...")

    result = agent.invoke(
        Command(
            resume={
            "decisions": [
                {"type":"approved"}
                ]
            }
        ),
        config=config
    )

    print(f"result:{result['messages'][-1].content}")

paused! Approved...


ValueError: Unexpected human decision: {'type': 'approved'}. Decision type 'approved' is not allowed for tool 'send_email_tool'. Expected one of ['approved', 'edit', 'reject'] based on the tool's configuration.